# Revenue Lift Proposal — Two-Model Analysis (Churn + Revenue)

**Client:** Company A (telecom) · **Deliverable notebook** for the GCI World 2026 final assignment

**Business question:** *Where is revenue leaking through churn, and which customer
variables, if acted on, produce the largest revenue lift?*

This notebook builds **two models and then combines them**:

| | Model | Target | Why the proposal needs it |
|---|---|---|---|
| **A** | XGBoost binary classifier | `churn` | *Who is about to leave?* → who to contact |
| **B** | XGBoost regressor | `rev_Mean` (monthly revenue) | *What is each customer worth?* → what is at stake |

`Model A probability × Model B predicted value = **revenue-at-risk**`, a per-customer
priority score. A campaign simulation then quantifies the lift of targeting on that
score versus targeting on churn probability alone or at random.

**Sections**

1. Setup and data
2. EDA recap (deck figures 01–04)
3. Feature engineering (deck figure 05)
4. Leakage audit and model-specific feature sets
5. Model A — churn (deck figures 06–07)
6. Model B — revenue (deck figures 08–09)
7. Combining the models: revenue-at-risk (deck figure 10)
8. Campaign simulation and lift (deck figures 11–13)
9. Results box for the slides, assumptions register, proposal statement
10. References

Every figure used in the slide deck is written to `assets/` as a `200 dpi` PNG whose
file name starts with its slide order number.

## 0. Setup

Run this notebook in **Google Colab** (recommended) or any Jupyter environment with
`pandas`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn` installed.

In Colab:

1. Mount Drive (the next cell does it automatically when it detects Colab).
2. Uncomment the `os.chdir(...)` line and point it at the folder that contains
   `telecom/`, `tutorial.ipynb` and this notebook.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive


from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error, r2_score,
)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="talk")

# --- Colab only -----------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # >>> change this to the folder that holds telecom/ <<<
    # os.chdir("/content/drive/MyDrive/Final Assignment")

# --- project + assets folders ---------------------------------------------------
def find_project_dir(start: Path) -> Path:
    # Walk upwards until we find the folder containing telecom/Client.csv
    for candidate in [start, *start.parents]:
        if (candidate / "telecom" / "Client.csv").exists():
            return candidate
    return start

DRIVE_ROOT = Path("/content/drive/MyDrive")

matches = [
    p / "telecom" / "Client.csv"
    for p in DRIVE_ROOT.iterdir()
    if p.is_dir() and (p / "telecom" / "Client.csv").exists()
]

if len(matches) != 1:
    raise RuntimeError(f"Expected one project, found {len(matches)}")

PROJECT_DIR = matches[0].parent.parent
ASSETS = PROJECT_DIR / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

def show(fig, name):
    # Save a figure to assets/ (200 dpi) and display it.
    path = ASSETS / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"[assets] {name}.png")
    plt.show()
    return path

print("project :", PROJECT_DIR)
print("assets  :", ASSETS)
print("colab   :", IN_COLAB)

## 0.1 Market context — why this proposal, why now

Three facts frame the business case; each carries a citation that must appear on the
slide where it is used:

* **Growth now has to come from the installed base.** GSMA's *The Mobile Economy
  2026* reports that mobile technologies generated **\$7.6 trillion — 6.4% of global
  GDP — in 2025**, and that **45% of operators now name revenue monetisation a
  strategic priority**. In a mature market, monetisation means more revenue *per
  existing subscriber*, not more subscribers (GSMA, 2026).
* **Retention is the cheapest revenue lever.** Acquiring a new customer costs
  **5–25× more than keeping an existing one**, and Bain's Reichheld research finds
  that a **5% rise in retention increases profits by 25–95%**
  (Harvard Business Review, 2014).
* **Company A has ~100,000 customers and no in-house modelling capability.** It
  cannot say *which* customers are leaving or *what each is worth* — the two numbers
  you need before spending a retention budget.

That is the gap this notebook closes: two models that turn raw usage and billing
history into a ranked, dollar-denominated list of customers worth defending.

> Sources: GSMA, *The Mobile Economy 2026*; Gallo, A., *The Value of Keeping the
> Right Customers*, HBR (2014). Full reference list in Section 11.

## 1. Data

Two tables, one row per customer, joined 1:1 on `Customer_ID`:

* `Record.csv` — average monthly usage, billing and call-quality metrics **plus the `churn` target**
* `Client.csv` — account, demographics, equipment and household attributes

In [ ]:
client = pd.read_csv(PROJECT_DIR / "telecom" / "Client.csv")
record = pd.read_csv(PROJECT_DIR / "telecom" / "Record.csv")

df = record.merge(client, on="Customer_ID", how="inner")

print(f"Client {client.shape} | Record {record.shape} -> merged {df.shape}")
print(f"duplicated Customer_ID after merge: {df['Customer_ID'].duplicated().sum()}")
df.head(3)

## 2. EDA recap

The tutorial notebook already covered shape, types and missingness. Here we keep the
four views the proposal actually needs, because they set up everything that follows:

1. **Class balance** — decides whether accuracy is a meaningful metric.
2. **Revenue vs churn** — decides whether churn is a *revenue* problem at all.
3. **Missingness** — decides the preprocessing policy.
4. **Correlations** — decides which features are duplicates and where leakage hides.

In [ ]:
# ---- 01: class balance --------------------------------------------------------
counts = df["churn"].value_counts().sort_index()
labels = ["stayed (0)", "churned (1)"]
pcts = (counts / counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(labels, counts.values, color=["#4C72B0", "#C44E52"], width=0.55)
for b, n, p in zip(bars, counts.values, pcts.values):
    ax.text(b.get_x() + b.get_width() / 2, n + counts.max() * 0.02,
            f"{n:,}\n{p}%", ha="center", va="bottom", fontsize=13)
ax.set_ylabel("Customers")
ax.set_title("01 — Churn is close to balanced (50/50), not the usual 5-10%")
ax.set_ylim(0, counts.max() * 1.22)
sns.despine(ax=ax)

show(fig, "01_churn_balance")

A ~50/50 split is unusual for telecom churn (real bases run 5–10%). Two consequences:

* **Accuracy is legitimate here** — it is not inflated by a majority class, and a
  "always predict stay" model would score 0.50. We therefore report accuracy *and*
  ROC-AUC, and we still show the confusion matrix because the two error types have
  very different costs.
* **The model has plenty of positive examples to learn from**, so a 70/30 split is
  enough; no resampling or class weighting is needed.

In [ ]:
# ---- 02: revenue vs churn -----------------------------------------------------
means = df.groupby("churn")["rev_Mean"].mean()
print("Mean monthly revenue")
print(means.round(2).rename(index={0: "stayed", 1: "churned"}))
print(f"Difference: ${means[1] - means[0]:.2f} "
      f"({(means[1] / means[0] - 1) * 100:+.1f}%)")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x="churn", y="rev_Mean", hue="churn", showfliers=False,
            palette=["#4C72B0", "#C44E52"], width=0.5, ax=ax)
if ax.legend_ is not None:
    ax.legend_.remove()
ax.axhline(means[0], color="#4C72B0", ls="--", lw=1.5)
ax.axhline(means[1], color="#C44E52", ls="--", lw=1.5)
ax.text(0.42, means.max() * 0.97,
        f"mean stayed  ${means[0]:.2f}\nmean churned ${means[1]:.2f}\n"
        f"gap {(means[1] / means[0] - 1) * 100:+.1f}%",
        transform=ax.get_yaxis_transform(), fontsize=12, va="top")
ax.set_xticks([0, 1])
ax.set_xticklabels(["stayed (0)", "churned (1)"])
ax.set_xlabel("")
ax.set_ylabel("Average monthly revenue ($)")
ax.set_title("02 — Churners spend the same as stayers: the loss is volume, not price")
sns.despine(ax=ax)

show(fig, "02_revenue_by_churn")

**This is the finding the whole proposal rests on.** Churners and stayers spend
almost exactly the same, so *churn is not concentrated among big spenders*. A
"save our best customers" campaign built on revenue alone would target the wrong
people, and a campaign built on churn probability alone would spend money saving
customers worth \$5 or \$500 with equal enthusiasm.

The lift has to come from the **intersection**: customers who are *both* likely to
churn *and* worth a lot. That score is exactly `p(churn) × revenue` — which is why
the proposal needs two models rather than one.

In [ ]:
# ---- 03: missing values -------------------------------------------------------
miss = df.isna().sum()
miss_pct = miss / len(df) * 100
top = miss_pct.sort_values(ascending=False).head(20)

print(f"Columns with any missing value: {(miss > 0).sum()} of {df.shape[1]}")
print(f"Columns above 40% missing: {list(miss_pct[miss_pct > 40].index)}")

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#C44E52" if v > 40 else "#4C72B0" for v in top.values]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
for i, v in enumerate(top.values[::-1]):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=11)
ax.set_xlabel("% of customers missing")
ax.set_title("03 — Missingness is concentrated in the household/demographic block")
ax.set_xlim(0, top.max() * 1.18)
sns.despine(ax=ax)

show(fig, "03_missing_values")

**Preprocessing policy that follows from this figure**

* Only `numbcars` exceeds 40% missing. Dropping it loses little and removes a column
  that would otherwise be mostly noise — we drop it.
* `dwllsize`, `HHstatin`, `ownrent`, `dwlltype`, `lor`, `income`, `adults`,
  `infobase` sit between 20–40%. These are household attributes that are often simply
  not collected. We **keep them and treat "missing" as its own category** rather than
  imputing a guess: for a tree model an absent value is itself an observable signal
  (customers who never disclosed income behave differently), and XGBoost routes
  missing values down the tree while learning.
* The `Record.csv` usage block is only 0.4% missing and `change_mou`/`change_rev`
  0.9% — negligible, left as `NaN` for the same reason.
* No column is dropped for missingness other than `numbcars`, and no mean/mode
  imputation is used anywhere. This is a deliberate, documented choice.

In [ ]:
# ---- 04: correlations among business-relevant features ------------------------
key_cols = [
    "rev_Mean", "mou_Mean", "totmrc_Mean", "ovrmou_Mean", "avg6mou",
    "totrev", "eqpdays", "months", "custcare_Mean", "drop_vce_Mean",
]
corr = df[key_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"fontsize": 11}, ax=ax)
ax.set_title("04 — Revenue, usage and plan price move together\n(r = Pearson, key features only)")

show(fig, "04_correlation_heatmap")

Two readings that matter downstream:

* `mou_Mean` (minutes used) and `avg6mou` sit at 0.60–0.70 with `rev_Mean`, while
  `totrev`/`avg6rev` sit at 0.52–0.86. **Past-total revenue is almost the same
  variable as current revenue** — using it as a feature would hand the revenue model
  the answer. Section 4 quantifies this and excludes those columns.
* Nothing is perfectly collinear at feature level, so there is no need for PCA or
  feature dropping for multicollinearity: tree models are unaffected by it anyway.

## 3. Feature engineering

Raw columns describe *what happened*. The proposal needs features that describe
*what is going wrong*, because only those name an actionable lever. Each group below
is created for a specific business reason:

| Group | Features | Business question it answers |
|---|---|---|
| **Usage trend** | `usage_trend_mou`, `usage_trend_rev` | Is the customer using less than they did six months ago? |
| **Overage economics** | `overage_share_mou`, `overage_share_rev`, `overage_months_flag` | Are they paying penalties outside their plan (a price shock and a churn trigger)? |
| **Service quality** | `failed_call_rate`, `call_completion`, `care_calls_per_100min` | Is the network or support experience driving them away? |
| **Equipment** | `equip_age_months`, `equip_age_vs_tenure`, `handset_upgrade_gap` | Is an ageing handset making renewal unattractive? |
| **Household & plan** | `models_per_phone`, `tenure_bucket`, `rev_tier` | How complex/sticky is the account? |

`NaN` is preserved: a ratio is undefined when the denominator is zero, and zero
denominators are themselves a customer type (customers who never called).

In [ ]:
def safe_div(a, b):
    # 0 / 0 and x / 0 become NaN: undefined, not zero. Trees handle NaN natively.
    return np.where(b > 0, a / b, np.nan)

f = pd.DataFrame(index=df.index)

# --- usage trend: last 3 months vs last 6 months -------------------------------
f["usage_trend_mou"] = (df["avg3mou"] - df["avg6mou"]) / df["avg6mou"].replace(0, np.nan)
f["usage_trend_rev"] = (df["avg3rev"] - df["avg6rev"]) / df["avg6rev"].replace(0, np.nan)
f["usage_trend_qty"] = (df["avg3qty"] - df["avg6qty"]) / df["avg6qty"].replace(0, np.nan)

# --- overage economics ---------------------------------------------------------
f["overage_share_mou"] = safe_div(df["ovrmou_Mean"], df["mou_Mean"])
f["overage_share_rev"] = safe_div(df["ovrrev_Mean"], df["rev_Mean"])
f["overage_flag"] = (df["ovrmou_Mean"].fillna(0) > 0).astype(int)

# --- service quality -----------------------------------------------------------
placed = df["plcd_vce_Mean"].replace(0, np.nan)
f["call_completion"] = safe_div(df["comp_vce_Mean"], placed)
f["failed_call_rate"] = 1 - f["call_completion"]
f["call_drop_rate"] = safe_div(df["drop_vce_Mean"], placed)
f["care_calls_per_100min"] = safe_div(df["custcare_Mean"] * 100, df["mou_Mean"])

# --- equipment ------------------------------------------------------------------
f["equip_age_months"] = df["eqpdays"] / 30.44
f["equip_age_vs_tenure"] = f["equip_age_months"] - df["months"]
f["handset_upgrade_gap"] = np.where(df["eqpdays"] > df["months"] * 30.44, 1, 0)

# --- household / plan -----------------------------------------------------------
f["models_per_phone"] = safe_div(df["models"], df["phones"])
f["lines_per_customer"] = df["uniqsubs"]
f["rev_per_mou"] = safe_div(df["rev_Mean"], df["mou_Mean"])
f["minutes_per_line"] = safe_div(df["mou_Mean"], df["uniqsubs"])

# --- ordinal business buckets ---------------------------------------------------
f["tenure_bucket"] = pd.cut(
    df["months"], bins=[0, 6, 12, 24, 48, 1000],
    labels=[0, 1, 2, 3, 4]).astype(float)
f["rev_tier"] = pd.qcut(df["rev_Mean"].rank(method="first"), 4,
                        labels=[0, 1, 2, 3]).astype(float)

df = pd.concat([df, f], axis=1)

print(f"Dataset: {df.shape[0]:,} customers x {df.shape[1]} columns "
      f"({f.shape[1]} engineered)")
df[list(f.columns)].describe().T.round(3)

In [ ]:
# ---- 05: do the engineered signals actually separate the classes? -------------
check_cols = ["usage_trend_mou", "overage_share_mou", "failed_call_rate",
              "equip_age_months"]
plot_df = df[check_cols + ["churn"]].melt(id_vars="churn", var_name="feature",
                                          value_name="value")

plot_df["feature"] = plot_df["feature"].map({
    "usage_trend_mou": "Usage trend\n(3m vs 6m, % change)",
    "overage_share_mou": "Overage share\n(% of minutes outside plan)",
    "failed_call_rate": "Failed call rate\n(calls not completed)",
    "equip_age_months": "Equipment age\n(months)",
})
plot_df["class"] = plot_df["churn"].map({0: "stayed", 1: "churned"})

g = sns.FacetGrid(plot_df, col="feature", sharex=False, height=4.2, aspect=1.05)
g.map_dataframe(sns.boxplot, x="class", y="value",
                order=["stayed", "churned"], hue="class",
                palette=["#4C72B0", "#C44E52"], showfliers=False)
g.set_axis_labels("", "value")
g.set(xticks=[0, 1], xticklabels=["stayed", "churned"])
g.set_titles("{col_name}", fontsize=13)
for ax in g.axes.flat:
    if ax.legend_ is not None:
        ax.legend_.remove()
g.fig.suptitle("05 — Engineered signals separate churners from stayers",
               fontsize=16, y=1.03)

show(g.fig, "05_engineered_signals_vs_churn")

Read the boxes: churners show a more negative usage trend, a different overage
profile, more failed calls and older equipment. None of them separates the classes
on its own — churn is multi-factorial, which is precisely the argument for a
gradient-boosted model over a single-variable rule, and for *combining* signals into
one priority score rather than acting on any single one.

## 4. Leakage audit and model-specific feature sets

The two models do **not** share an identical feature set, and the difference has to
be justified rather than assumed.

**Churn model.** Everything except `Customer_ID` and the `churn` label. Current
revenue, usage and equipment age are all observed *before* the customer leaves, so
they are legitimate predictors.

**Revenue model.** The target is `rev_Mean` — average monthly revenue. Several
columns are other aggregations or additive components of that same quantity:

| Excluded | Reason |
|---|---|
| `avg6rev`, `avg3rev`, `avgrev`, `totrev`, `adjrev` | trailing/total revenue — same measurement, different window |
| `ovrrev_Mean`, `vceovr_Mean`, `datovr_Mean`, `roam_Mean`, `da_Mean` | additive components of `rev_Mean` (revenue = recurring + overage + roaming + data) |
| `change_rev`, `usage_trend_rev`, `overage_share_rev`, `rev_per_mou` | computed from the target itself |
| `totmrc_Mean` | the contracted recurring charge: essentially the price of the target |
| `churn` | unknown at scoring time |

Keeping any of them would give a near-perfect R² that means nothing. The audit
below measures how bad the naive version is before we commit to the honest one.

In [ ]:
ID_COL, TARGET_CHURN, TARGET_REV = "Customer_ID", "churn", "rev_Mean"

categorical_cols = [c for c in df.columns if df[c].dtype == "object"]
REV_LEAK = [
    "avg6rev", "avg3rev", "avgrev", "totrev", "adjrev",
    "ovrrev_Mean", "vceovr_Mean", "datovr_Mean", "roam_Mean", "da_Mean",
    "change_rev", "usage_trend_rev", "overage_share_rev", "rev_per_mou",
    "totmrc_Mean", "churn",
]

churn_cols = [c for c in df.columns if c not in (ID_COL, TARGET_CHURN)]
rev_naive_cols = [c for c in df.columns if c not in (ID_COL, TARGET_REV, TARGET_CHURN)]
rev_cols = [c for c in rev_naive_cols if c not in REV_LEAK]

def make_X(cols):
    X = df[cols].copy()
    for c in categorical_cols:
        if c in X.columns:
            # NaN becomes its own category code; numeric NaN stays NaN for XGBoost
            X[c] = pd.factorize(X[c].astype(str))[0]
    return X

print(f"churn features     : {len(churn_cols)}")
print(f"revenue features   : {len(rev_cols)}  (naive: {len(rev_naive_cols)})")
print(f"categorical columns: {len(categorical_cols)} -> {categorical_cols}")
print(f"excluded from revenue model: {sorted(set(rev_naive_cols) - set(rev_cols))}")

## 5. Model A — Churn (XGBoost binary classifier)

**Why this model.** The data is tabular, ~100 columns, mixed scales, label-encoded
categoricals and genuine missing values. XGBoost needs no scaling, handles `NaN` by
learning default directions for it, resists overfitting through row/column
subsampling, and — decisively for this assignment — reports **gain-based feature
importance we can turn into business levers**. A logistic regression would assume
linear log-odds that these engineered ratios do not satisfy; a random forest would
give the same structure without the regularisation control.

**Evaluation protocol.** Stratified 70/30 split, `random_state=42`. Churn is ~50%,
so accuracy is *not* inflated by a majority class: an "always predict stayed" model
scores 0.50, which is the number to beat. ROC-AUC is the headline metric because it
is threshold-free.

**The threshold is a business decision, not a modelling decision.** Nothing in the
model says whether 0.50 is the right cut-off: that depends on how many customers the
retention team can actually contact. Metrics are therefore reported at 0.50 for
comparability, and the confusion matrix is read alongside the *campaign operating
point* — the top 10% of scores, which is what Section 8 spends money on.

In [ ]:
X_churn = make_X(churn_cols)
y_churn = df[TARGET_CHURN].astype(int)

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_churn, y_churn, test_size=0.30, random_state=RANDOM_STATE, stratify=y_churn)

churn_model = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_lambda=1.0, eval_metric="logloss", tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1,
)
churn_model.fit(Xc_tr, yc_tr)

p_test = churn_model.predict_proba(Xc_te)[:, 1]
y_pred = (p_test >= 0.5).astype(int)          # default operating point

acc = accuracy_score(yc_te, y_pred)
auc = roc_auc_score(yc_te, p_test)
f1 = f1_score(yc_te, y_pred)

# Threshold is a business decision, not a modelling one: it depends on how many
# customers the campaign can actually contact. We therefore also report what the
# model delivers at the campaign contact budget (top 10% of scores = figure 11).
CONTACT_FRAC = 0.10
budget_cut = np.quantile(p_test, 1 - CONTACT_FRAC)
in_budget = p_test >= budget_cut
recall_at_budget = ((in_budget) & (yc_te == 1)).sum() / (yc_te == 1).sum()
precision_at_budget = ((in_budget) & (yc_te == 1)).sum() / in_budget.sum()

print(f"train {Xc_tr.shape[0]:,} | test {Xc_te.shape[0]:,}")
print(f"accuracy  : {acc:.4f}   (baseline always-'stayed' = "
      f"{1 - yc_te.mean():.4f})")
print(f"ROC-AUC   : {auc:.4f}")
print(f"F1        : {f1:.4f}  @ threshold 0.50")
print()
print(classification_report(yc_te, y_pred, target_names=["stayed", "churned"]))
print(f"At the campaign contact budget (top {CONTACT_FRAC:.0%} of risk scores, "
      f"cutoff p >= {budget_cut:.2f}):")
print(f"  recall of actual churners : {recall_at_budget:.1%}")
print(f"  precision in the batch     : {precision_at_budget:.1%} "
      f"(vs {yc_te.mean():.1%} base rate)")

In [ ]:
# ---- 06: evaluation ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

fpr, tpr, thr = roc_curve(yc_te, p_test)
axes[0].plot(fpr, tpr, color="#4C72B0", lw=2.5,
             label=f"Model A  AUC = {auc:.3f}")
axes[0].plot([0, 1], [0, 1], ls="--", c="grey", lw=1.5, label="random (AUC 0.50)")
# the campaign operating point: top 10% of scores
j = int(np.argmin(np.abs(thr - budget_cut)))
axes[0].plot(fpr[j], tpr[j], "o", ms=11, c="#C44E52",
             label=f"campaign budget (top {CONTACT_FRAC:.0%} of scores)\n"
                  f"catches {tpr[j]:.0%} of churners,\n"
                  f"{fpr[j]:.0%} false-alarm rate")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].legend(loc="lower right", fontsize=11)
axes[0].set_aspect("equal")

cm = confusion_matrix(yc_te, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["predicted stay", "predicted churn"],
            yticklabels=["actual stay", "actual churn"],
            annot_kws={"fontsize": 14}, ax=axes[1])
axes[1].set_title("Confusion matrix at threshold 0.50")

fig.suptitle("06 — Model A: XGBoost churn classifier, "
             f"accuracy {acc:.3f} / ROC-AUC {auc:.3f}", fontsize=16, y=1.02)
fig.tight_layout()
show(fig, "06_churn_model_eval")

### 7.1 Where the churn signal comes from

`gain` is used rather than `weight`: a feature can be split on constantly while
contributing almost nothing, whereas gain measures how much each split improves the
model. Importance alone is not actionable, so each top feature is tagged as either an
**intervenable lever** (the business can change it: offers, equipment, support,
overage plan) or a **fixed attribute** (tenure, demographics — useful for
segmentation, useless as an action).

In [ ]:
# ---- 07: feature importance (gain) --------------------------------------------
imp = pd.Series(churn_model.get_booster().get_score(importance_type="gain"))

# get_score() returns keys like f0, f1 ... -> map them back to real column names
if all(str(i).startswith("f") for i in imp.index):
    imp = imp.rename(index={f"f{i}": c for i, c in enumerate(churn_cols)})
imp = imp.groupby(level=0).mean().sort_values(ascending=False)

top15 = imp.head(15).sort_values()

LEVERS = {
    "equip_age_months": "lever", "equip_age_vs_tenure": "lever",
    "handset_upgrade_gap": "lever", "eqpdays": "lever",
    "failed_call_rate": "lever", "call_drop_rate": "lever",
    "call_completion": "lever", "drop_vce_Mean": "lever",
    "overage_share_mou": "lever", "overage_share_rev": "lever",
    "overage_flag": "lever", "ovrmou_Mean": "lever", "ovrrev_Mean": "lever",
    "custcare_Mean": "lever", "care_calls_per_100min": "lever",
    "usage_trend_mou": "lever", "usage_trend_rev": "lever",
    "usage_trend_qty": "lever", "usage_trend": "lever",
    "avg3mou": "lever", "avg3qty": "lever", "change_mou": "lever",
    "hnd_webcap": "lever", "hnd_price": "lever", "refurb_new": "lever",
    "rev_Mean": "price lever", "totmrc_Mean": "price lever",
}
tags = [LEVERS.get(name, "fixed / context") for name in top15.index]

fig, ax = plt.subplots(figsize=(10, 7.5))
colors = ["#4C72B0" if t == "fixed / context" else "#DD8452" for t in tags]
ax.barh(top15.index, top15.values, color=colors)
for i, (v, t) in enumerate(zip(top15.values, tags)):
    ax.text(v * 1.02, i, t, va="center", fontsize=10, color="#333333")
ax.set_xlabel("Mean gain per split")
ax.set_title("07 — Model A feature importance (gain)\n"
             "orange = actionable lever,  blue = fixed customer attribute")
ax.set_xlim(0, top15.max() * 1.35)
sns.despine(ax=ax)
fig.tight_layout()
show(fig, "07_churn_importance")

lever_share = np.mean([t != "fixed / context" for t in tags])
print(f"\nActionable levers inside the top 15: {int(lever_share * 15)}/15")
display(pd.DataFrame({"feature": top15.index[::-1],
                      "mean_gain": top15.values[::-1].round(1),
                      "type": tags[::-1]}))

## 6. Model B — Revenue (XGBoost regressor)

**Target:** `rev_Mean`, the customer's average monthly revenue — the quantity a
retained customer keeps producing. Rows where `rev_Mean` is missing (357 customers)
are dropped for this model only.

**Why a regressor and not a classifier.** Binning revenue into "high / low" would
throw away the exact information the lift calculation multiplies out. We need a
per-customer dollar figure, so we predict it directly.

**Two versions are trained on purpose:**

* *naive* — everything except the target and the ID: a deliberately generous feature
  set that still contains trailing-revenue aggregates;
* *honest* — the leakage-audited set from Section 4.

Reporting both shows the score that is really achievable, and the gap between them
is itself evidence that the honest model is doing genuine work. A mean-predictor
baseline is included so the metrics mean something.

In [ ]:
df_rev = df[df[TARGET_REV].notna()].copy()
print(f"Revenue model rows: {len(df_rev):,} (dropped {df.isna()[TARGET_REV].sum():,} "
      f"customers with no revenue record)")

y_rev = df_rev[TARGET_REV].astype(float)

def fit_revenue(cols, label):
    X = make_X(cols).loc[df_rev.index]
    Xtr, Xte, ytr, yte = train_test_split(
        X, y_rev, test_size=0.30, random_state=RANDOM_STATE)
    model = XGBRegressor(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_lambda=1.0, objective="reg:squarederror", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    res = {
        "variant": label,
        "n_features": X.shape[1],
        "MAE": mean_absolute_error(yte, pred),
        "RMSE": float(np.sqrt(mean_squared_error(yte, pred))),
        "R2": r2_score(yte, pred),
        "baseline_MAE": mean_absolute_error(yte, np.repeat(ytr.mean(), len(yte))),
        "model": model, "X_test": Xte, "y_test": yte, "pred": pred,
    }
    return res

rev_naive = fit_revenue(rev_naive_cols, "B1 naive (leaky features kept)")
rev_honest = fit_revenue(rev_cols, "B2 honest (leakage-audited)")

results = pd.DataFrame([{
    k: v for k, v in r.items() if k in
    ("variant", "n_features", "MAE", "RMSE", "R2", "baseline_MAE")
} for r in (rev_naive, rev_honest)]).round(3)
results

The gap between B1 and B2 **is** the leakage audit: `avg6rev` alone correlates 0.86
with the target, so B1's near-perfect fit is mostly recalling the answer. **B2 is
the model the proposal uses** — it predicts revenue from behaviour, plan
composition, equipment and demographics, i.e. from things a manager can influence
or segment on.

In [ ]:
# ---- 08: honest revenue model fit ---------------------------------------------
best = rev_honest
pred, yte = best["pred"], best["y_test"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

hb = axes[0].hexbin(yte, pred, gridsize=60, cmap="viridis", bins="log")
axes[0].plot([0, yte.max()], [0, yte.max()], ls="--", c="red", lw=1.5)
axes[0].set_xlabel("Actual monthly revenue ($)")
axes[0].set_ylabel("Predicted monthly revenue ($)")
axes[0].set_title(f"Predicted vs actual   (R2 = {best['R2']:.3f})")
fig.colorbar(hb, ax=axes[0], label="log10 customers")

resid = pred - yte.to_numpy()
axes[1].hist(resid, bins=80, color="#4C72B0")
axes[1].axvline(0, c="red", ls="--", lw=1.5)
axes[1].axvline(np.mean(resid), c="orange", ls="-", lw=1.5)
axes[1].set_xlabel("Prediction error ($)")
axes[1].set_ylabel("Customers")
axes[1].set_title(f"Residuals   mean {resid.mean():+.2f}, std {resid.std():.2f}")

fig.suptitle("08 — Model B (XGBoost regressor) fit on held-out data\n"
             f"MAE ${best['MAE']:.2f} vs baseline ${best['baseline_MAE']:.2f} "
             "when always predicting the average",
             fontsize=15, y=1.03)
fig.tight_layout()
show(fig, "08_revenue_model_fit")

In [ ]:
# ---- 09: revenue model importance --------------------------------------------
imp_r = pd.Series(best["model"].get_booster().get_score(importance_type="gain"))
if all(str(i).startswith("f") for i in imp_r.index):
    imp_r = imp_r.rename(index={f"f{i}": c for i, c in enumerate(rev_cols)})
imp_r = imp_r.groupby(level=0).mean().sort_values(ascending=False)

top15r = imp_r.head(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.barh(top15r.index, top15r.values, color="#55A868")
for i, v in enumerate(top15r.values):
    ax.text(v * 1.02, i, f"{v:,.0f}", va="center", fontsize=10)
ax.set_xlabel("Mean gain per split")
ax.set_title("09 — Model B feature importance (gain): what actually drives\n"
             "monthly revenue per customer")
ax.set_xlim(0, top15r.max() * 1.30)
sns.despine(ax=ax)
fig.tight_layout()
show(fig, "09_revenue_importance")

## 7. Combining the models: revenue-at-risk

$$\text{revenue-at-risk}_i \;=\; \hat{p}(\text{churn}_i)\;\times\;\widehat{\text{rev}}_i$$

After honest holdout evaluation both models are **refit on all data** so that every
customer in the base gets a score — the metrics above remain the honest ones, this
refit is only for scoring.

The score is validated against *realised* money: among customers who actually
churned, how much of their revenue does the top *x%* of the score capture? A score
that beats (a) random selection and (b) churn probability alone is worth paying for.

In [ ]:
# refit on all rows for scoring
churn_full = XGBClassifier(**{**churn_model.get_params(), "n_jobs": -1})
churn_full.fit(X_churn, y_churn)
df["p_churn"] = churn_full.predict_proba(X_churn)[:, 1]

rev_cols_X = make_X(rev_cols).loc[df_rev.index]
rev_full = XGBRegressor(**{**best["model"].get_params(), "n_jobs": -1})
rev_full.fit(rev_cols_X, y_rev)
df["rev_pred"] = rev_full.predict(make_X(rev_cols))

df["revenue_at_risk"] = df["p_churn"] * df["rev_pred"]
df["churned_revenue"] = df["churn"] * df["rev_Mean"]

sim = df.dropna(subset=["rev_Mean", "revenue_at_risk"]).copy()
print(f"Simulation base: {len(sim):,} customers")
print(f"Total monthly revenue      : ${sim['rev_Mean'].sum():,.0f}")
print(f"Expected monthly churn loss: ${sim['revenue_at_risk'].sum():,.0f} "
      f"({sim['revenue_at_risk'].sum() / sim['rev_Mean'].sum():.1%} of revenue)")
print(f"Realised monthly churn loss: ${sim['churned_revenue'].sum():,.0f}")
sim[["Customer_ID", "p_churn", "rev_pred", "revenue_at_risk"]].head()

In [ ]:
# ---- 10: does the score capture real churned revenue? --------------------------
N = len(sim)
k_grid = np.arange(0.05, 1.01, 0.01)

def gains(score_col):
    order = sim.sort_values(score_col, ascending=False)
    realised = order["churned_revenue"].to_numpy()
    total = realised.sum()
    cum = np.cumsum(realised) / total
    return cum

percents = np.arange(1, N + 1) / N
g_risk = gains("revenue_at_risk")
g_churn = gains("p_churn")
g_rev = gains("rev_pred")
rng = np.random.default_rng(RANDOM_STATE)
g_random = np.cumsum(
    rng.permutation(sim["churned_revenue"].to_numpy())) / sim["churned_revenue"].sum()

def lift_at(curve, frac):
    i = int(frac * N) - 1
    return curve[max(i, 0)]

for f in (0.1, 0.2, 0.3):
    print(f"Top {f:.0%}: revenue-at-risk captures {lift_at(g_risk, f):.1%} "
          f"| churn-prob only {lift_at(g_churn, f):.1%} "
          f"| revenue only {lift_at(g_rev, f):.1%} | random {f:.0%}")

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.plot(percents * 100, g_random * 100, ls="--", c="grey", lw=1.8,
        label="random targeting")
ax.plot(percents * 100, g_rev * 100, ls=":", c="#55A868", lw=2.2,
        label="target by revenue only (Model B)")
ax.plot(percents * 100, g_churn * 100, ls="--", c="#C44E52", lw=2.2,
        label="target by churn probability (Model A)")
ax.plot(percents * 100, g_risk * 100, c="#4C72B0", lw=3,
        label="target by revenue-at-risk (A x B)")
ax.plot([0, 100], [0, 100], c="k", lw=1)
ax.set_xlabel("% of customer base contacted (highest score first)")
ax.set_ylabel("% of realised churned revenue captured")
ax.set_title("10 — Combining both models captures churned revenue faster\n"
             "(gains chart on customers who actually churned)")
ax.legend(loc="lower right", fontsize=11)
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
sns.despine(ax=ax)
fig.tight_layout()
show(fig, "10_risk_deciles_gains")

## 8. Campaign simulation and revenue lift

The simulation answers the client's real question: *if we run a retention campaign
on the customers the models flag, how much net revenue do we gain, and which
variables decide the outcome?*

### Assumptions (all editable constants below)

| Assumption | Value | Rationale |
|---|---|---|
| Contact cost | 2.00 USD / customer contacted | outbound call + handling |
| Contact rate | 80% | share of targeted customers actually reached |
| Offer acceptance | 40% | share of reached customers who take the retention offer |
| Discount | 5% of monthly bill, while retained | typical retention-offer depth |
| Churn-reduction efficacy | 30% relative | share of an accepted customer's churn risk removed |
| Contact budget | top 10% of the base | ~10,000 outbound contacts |
| Horizon | 1 month, annualised ×12 | all figures are per campaign cycle |

**Cost logic:** the contact cost is paid for every customer we reach; the discount is
paid to every customer who accepts — including the ones who would have stayed
anyway. That cannibalisation is the honest cost of a retention offer and it is why
net lift is smaller than retained revenue.

$$\text{net lift}=\underbrace{\textstyle\sum \text{rev}_i\,p_i\,\text{CR}\,\text{AR}\,\epsilon}_{\text{revenue kept}}-\underbrace{k\cdot\text{CR}\cdot c_{\text{contact}}}_{\text{outreach}}-\underbrace{\textstyle\sum \text{rev}_i\,\text{CR}\,\text{AR}\,d}_{\text{discount}}$$

where CR = contact rate, AR = acceptance rate, $\epsilon$ = churn-reduction
efficacy, $d$ = discount depth, $k$ = number of customers contacted.

In [ ]:
# --- simulation parameters (edit these) -----------------------------------------
CONTACT_COST   = 2.00    # $ per customer we reach out to
CONTACT_RATE   = 0.80    # fraction of targeted customers reached
ACCEPT_RATE    = 0.40    # fraction of reached customers accepting the offer
DISCOUNT_PCT   = 0.05    # off the monthly bill, paid while they stay
EFFICACY       = 0.30    # relative reduction of churn risk for an accepting customer
TARGET_FRAC    = 0.10    # contact budget as a share of the customer base
MONTHS         = 12

K = int(len(sim) * TARGET_FRAC)

def simulate(score, k=K, efficacy=EFFICACY, discount=DISCOUNT_PCT,
             contact_rate=CONTACT_RATE, accept_rate=ACCEPT_RATE):
    # Net lift of contacting the k customers with the highest score.
    idx = np.argsort(-np.asarray(score))[:k]
    sub = sim.iloc[idx]
    accepted = contact_rate * accept_rate                 # reach x accept

    retained  = (sub["rev_Mean"] * sub["p_churn"] * accepted * efficacy).sum()
    outreach  = k * contact_rate * CONTACT_COST
    giveaway  = (sub["rev_Mean"] * accepted * discount).sum()
    cost      = outreach + giveaway
    net       = retained - cost
    return {
        "customers_contacted": k,
        "revenue_kept": retained,
        "contact_cost": outreach,
        "discount_cost": giveaway,
        "total_cost": cost,
        "net_monthly_lift": net,
        "net_annual_lift": net * MONTHS,
        "roi": net / cost if cost else np.nan,
        "index": idx,
    }

print(f"Contact budget: {K:,} customers "
      f"({TARGET_FRAC:.0%} of {len(sim):,})")

In [ ]:
# --- four scenarios --------------------------------------------------------------
random_scores = np.random.default_rng(RANDOM_STATE).random(len(sim))

scenarios = {
    "1. Do nothing (baseline)": None,
    "2. Random targeting": random_scores,
    "3. Churn probability only (Model A)": sim["p_churn"].to_numpy(),
    "4. Revenue-at-risk (Model A x Model B)": sim["revenue_at_risk"].to_numpy(),
}

rows = []
for name, score in scenarios.items():
    if score is None:
        rows.append({"scenario": name, "customers_contacted": 0,
                     "revenue_kept": 0.0, "contact_cost": 0.0,
                     "discount_cost": 0.0, "total_cost": 0.0,
                     "net_monthly_lift": 0.0, "net_annual_lift": 0.0,
                     "roi": np.nan})
    else:
        r = simulate(score)
        r.pop("index")
        r["scenario"] = name
        rows.append(r)

lift_table = pd.DataFrame(rows).set_index("scenario").round(2)
lift_table

In [ ]:
# ---- 11: lift by scenario -------------------------------------------------------
lt = lift_table.drop(index="1. Do nothing (baseline)")
plot = lt[["revenue_kept", "total_cost", "net_monthly_lift"]].reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(lt))
w = 0.26
cols = {"revenue_kept": "#55A868", "total_cost": "#C44E52",
        "net_monthly_lift": "#4C72B0"}
for i, c in enumerate(["revenue_kept", "total_cost", "net_monthly_lift"]):
    b = ax.bar(x + (i - 1) * w, plot[c], width=w, label=c.replace("_", " "),
               color=cols[c])
    ax.bar_label(b, fmt="$%.0f", fontsize=10, padding=2)

ax.set_xticks(x)
ax.set_xticklabels([s.split(". ", 1)[1] for s in lt.index], fontsize=12)
ax.set_ylabel("US$ per month")
ax.set_title("11 — Monthly campaign economics at a fixed contact budget\n"
             f"({K:,} customers contacted; annual net lift shown in the table below)")
ax.legend(fontsize=11, ncols=3, loc="upper left")
ax.set_ylim(0, plot[["revenue_kept", "total_cost"]].to_numpy().max() * 1.30)
sns.despine(ax=ax)
fig.tight_layout()
show(fig, "11_lift_vs_baseline")

tbl = lt[["revenue_kept", "total_cost", "net_monthly_lift",
          "net_annual_lift", "roi"]].round(0)
tbl["roi"] = lt["roi"].round(2)
print(tbl.to_string())

In [ ]:
# ---- 12: sensitivity — efficacy vs discount ------------------------------------
efficacies = np.linspace(0.05, 0.60, 12)
discounts = np.linspace(0.00, 0.15, 12)
scores = sim["revenue_at_risk"].to_numpy()

grid = np.zeros((len(discounts), len(efficacies)))
for i, d in enumerate(discounts):
    for j, e in enumerate(efficacies):
        grid[i, j] = simulate(scores, efficacy=e, discount=d)["net_monthly_lift"]

X, Y = np.meshgrid(efficacies, discounts)

from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(12, 7))
mesh = ax.pcolormesh(X, Y, grid / 1000, cmap="RdYlGn", shading="auto")
cs = ax.contour(X, Y, grid / 1000, levels=[0], colors="black", linewidths=2.5)
ax.clabel(cs, fmt=lambda lvl: "break-even", fontsize=11, inline=True)

for i, d in enumerate(discounts):
    for j, e in enumerate(efficacies):
        v = grid[i, j] / 1000
        ax.text(e, d, f"{v:,.0f}", ha="center", va="center", fontsize=9,
                color="black" if abs(v) < 100 else "white")

ax.plot(EFFICACY, DISCOUNT_PCT, marker="*", ms=26, mfc="gold", mec="black",
        mew=1.5, zorder=6, ls="none")
ax.annotate(f"base case\n{DISCOUNT_PCT:.0%} discount / {EFFICACY:.0%} efficacy\n"
            f"${simulate(scores)['net_monthly_lift']:,.0f} net / month",
            xy=(EFFICACY, DISCOUNT_PCT), xytext=(0.40, 0.012),
            fontsize=12, ha="center",
            bbox=dict(facecolor="white", edgecolor="black", alpha=0.92,
                      boxstyle="round,pad=0.45"),
            arrowprops=dict(arrowstyle="->", lw=1.8, color="black"))

cb = fig.colorbar(mesh, ax=ax, pad=0.02)
cb.set_label("net monthly lift ($ thousands)")
ax.set_xlabel("Churn-reduction efficacy (share of churn risk removed by the offer)")
ax.set_ylabel("Discount given on the monthly bill")
ax.set_title("12 — Net monthly lift: when is the campaign still profitable?\n"
             "green = profit, red = loss, black line = break-even")
ax.set_xlim(efficacies.min(), efficacies.max())
ax.set_ylim(discounts.min(), discounts.max())
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
fig.tight_layout()
show(fig, "12_sensitivity_heatmap")

be = []
for i, d in enumerate(discounts):
    row = grid[i]
    if (row > 0).any():
        be.append((d, efficacies[np.argmax(row > 0)]))
print("Break-even (minimum efficacy that still pays for itself):")
for d, e in be:
    print(f"  discount {d:5.1%} -> efficacy >= {e:5.1%}")

In [ ]:
# --- base rates: do these levers actually move churn and value? ------------------
ev = df.copy()
ev["equipment_band"] = pd.cut(ev["eqpdays"], bins=[-np.inf, 365, 548, 730, np.inf],
                              labels=["<1y", "1-1.8y", "1.8-2y", ">2y"])
ev["overage_band"] = pd.cut(ev["ovrmou_Mean"] / ev["mou_Mean"].replace(0, np.nan),
                            bins=[-np.inf, 0.05, 0.15, np.inf],
                            labels=["low (<5%)", "medium (5-15%)", "high (>15%)"],
                            ).fillna("low (<5%)")   # 0 minutes used = no overage
ev["tenure_band"] = pd.cut(ev["months"], bins=[-np.inf, 6, 12, 24, 48, np.inf],
                           labels=["<6m", "6-12m", "1-2y", "2-4y", ">4y"])

def band_table(col, name):
    g = ev.groupby(col, observed=True).agg(
        customers=("churn", "size"),
        churn_rate=("churn", "mean"),
        mean_revenue=("rev_Mean", "mean"),
    )
    g["revenue_at_risk_$"] = g["customers"] * g["churn_rate"] * g["mean_revenue"]
    g.index.name = name
    return g.round(3)

print("BASE RATES ON THE FULL CUSTOMER BASE")
print("=" * 74)
for col, name in [("equipment_band", "Equipment age"),
                  ("overage_band", "Overage exposure"),
                  ("tenure_band", "Tenure")]:
    t = band_table(col, name)
    print(f"\n--- {name} ---")
    print(t.to_string())

In [ ]:
# ---- 13: which variables decide the outcome -----------------------------------
idx = simulate(sim["revenue_at_risk"].to_numpy())["index"]
targeted = sim.iloc[idx].copy()
targeted["monthly_net_lift"] = (
    targeted["rev_Mean"] * targeted["p_churn"]
    * CONTACT_RATE * ACCEPT_RATE * EFFICACY
    - CONTACT_RATE * CONTACT_COST
    - targeted["rev_Mean"] * CONTACT_RATE * ACCEPT_RATE * DISCOUNT_PCT)

targeted["equipment_band"] = pd.cut(
    targeted["eqpdays"], bins=[-np.inf, 365, 548, 730, np.inf],
    labels=["<1y", "1-1.8y", "1.8-2y", ">2y"])
targeted["overage_band"] = pd.cut(
    targeted["overage_share_mou"].fillna(0), bins=[-np.inf, 0.05, 0.15, np.inf],
    labels=["no/low overage", "5-15%", ">15%"])
targeted["tenure_band"] = pd.cut(
    targeted["months"], bins=[-np.inf, 6, 12, 24, 48, np.inf],
    labels=["<6m", "6-12m", "1-2y", "2-4y", ">4y"])

segments = {
    "Equipment age": "equipment_band",
    "Overage exposure": "overage_band",
    "Tenure": "tenure_band",
    "Usage trend": pd.cut(targeted["usage_trend_mou"],
                          bins=[-np.inf, -0.2, 0.2, np.inf],
                          labels=["declining", "flat", "growing"]),
    "Area": "area",
}

rows = []
for label, col in segments.items():
    s = targeted.copy()
    s["_seg"] = s[col] if isinstance(col, str) else col
    g = s.groupby("_seg", observed=True).agg(
        customers=("Customer_ID", "count"),
        avg_risk=("revenue_at_risk", "mean"),
        net_lift=("monthly_net_lift", "sum"),
    )
    g = g.sort_values("net_lift", ascending=False)
    g.index.name = label
    rows.append(g)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (label, col), g in zip(axes[:3],
                               list(segments.items())[:3],
                               rows[:3]):
    palette = sns.color_palette("Blues_r", len(g))
    b = ax.bar(g.index.astype(str), g["net_lift"], color=palette)
    ax.bar_label(b, fmt="$%.0f", fontsize=10, padding=2)
    ax.set_title(label, fontsize=14)
    ax.set_ylabel("Net monthly lift ($)" if label == list(segments)[0] else "")
    ax.tick_params(axis="x", rotation=20)
    sns.despine(ax=ax)
fig.suptitle("13 — Where the lift comes from: net contribution by segment\n"
             f"(top {TARGET_FRAC:.0%} by revenue-at-risk, {K:,} customers)",
             fontsize=15, y=1.03)
fig.tight_layout()
show(fig, "13_segment_levers")

for g, name in zip(rows, segments):
    print(f"\n--- {name} ---")
    print(g.head(6).round(2).to_string())

### Reading figure 13

A segment earns a place in the offer design only if it passes three tests at once:

1. **Size** — enough customers to move the number;
2. **Concentration** — higher average revenue-at-risk than the campaign average;
3. **Intervenability** — the business can actually change the outcome.

The base-rate table above is the evidence, and it produces three clear conclusions:

* **Overage exposure passes all three.** Customers spending more than 15% of their
  minutes outside their plan are only **14.7% of the base** but churn at
  **56.1% versus 48.0%** for low-overage customers — *and* they spend **\$98.08 a
  month versus \$46.25**. They are simultaneously the most valuable and the most
  fragile group, which is exactly the intersection a single-variable campaign
  cannot see: 5,697 of the 9,964 customers we would contact (57%) sit in this band,
  against 14.7% of the base.
* **Equipment age passes tests 2 and 3 and is the strongest *directional* signal.***
  Churn rises monotonically with handset age, **45.2% under one year to 57.9% past
  two years**, and churners carry **421 days of equipment age versus 363** for
  stayers. It is Model A's top feature by gain (figure 07), and it is a lever the
  business pulls directly through device renewal.
* **Tenure passes test 1 only — use it for scheduling, not for the offer.** The
  **1–2 year** band is both the largest (42,868 customers) and the highest-churn
  (51.8%): a renewal-window campaign calendar, not something an agent can change on
  a call.
* **Usage trend fails the test — drop it.** The EDA hypothesis that falling usage
  predicts churn does *not* hold up in the scored population (the "declining" group
  shows *lower* average risk than the "growing" group). Reporting a lever that the
  data does not support would be the kind of thing a CFO can disprove in one
  question, so it is excluded from the recommendation.

**Resulting offer design:** (1) plan migration that removes the overage bill shock,
(2) handset renewal path for accounts over 18 months of equipment age, (3) timed to
the 1–2 year renewal window. Area splits are used for call-centre routing only.

## 9. Results box for the slides

The assignment requires the **model name, evaluation metric and score to be written
out explicitly** — copying the numbers below onto a slide satisfies that literally.

In [ ]:
lines = []
lines.append("=" * 74)
lines.append("MODEL RESULTS — copy onto the slides")
lines.append("=" * 74)
lines.append("MODEL A  |  XGBoost binary classifier (xgboost.XGBClassifier)")
lines.append(f"  target        : churn (0/1)")
lines.append(f"  split         : stratified 70/30, random_state=42, "
              f"n_test={len(yc_te):,}")
lines.append(f"  METRIC ROC-AUC: {auc:.4f}          <- headline")
lines.append(f"  METRIC accuracy: {acc:.4f}  (baseline {1 - yc_te.mean():.4f})")
lines.append(f"  METRIC F1     : {f1:.4f} @ threshold 0.50")
lines.append(f"  at campaign budget (top {CONTACT_FRAC:.0%} of scores): "
             f"recall of churners {recall_at_budget:.1%}, "
             f"precision {precision_at_budget:.1%}")
lines.append("-" * 74)
lines.append("MODEL B  |  XGBoost regressor (xgboost.XGBRegressor)")
lines.append("  target        : rev_Mean (average monthly revenue, $)")
lines.append(f"  split         : 70/30, random_state=42, n_test={len(yte):,}")
lines.append(f"  METRIC R2     : {best['R2']:.4f}   (naive/leaky version: "
              f"{rev_naive['R2']:.4f})")
lines.append(f"  METRIC MAE    : ${best['MAE']:.2f}  "
              f"(baseline, always predict the mean: ${best['baseline_MAE']:.2f})")
lines.append(f"  METRIC RMSE   : ${best['RMSE']:.2f}")
lines.append("-" * 74)
best_row = lt.loc["4. Revenue-at-risk (Model A x Model B)"]
alt = lt.loc["3. Churn probability only (Model A)"]
rand = lt.loc["2. Random targeting"]
lines.append("SIMULATION  |  top 10% by revenue-at-risk, "
             f"{K:,} customers contacted")
lines.append(f"  monthly net lift        : ${best_row['net_monthly_lift']:,.0f}")
lines.append(f"  annualised net lift     : ${best_row['net_annual_lift']:,.0f}")
lines.append(f"  campaign ROI            : {best_row['roi']:.2f}x")
lines.append(f"  vs churn-probability-only: "
              f"${alt['net_annual_lift']:,.0f} ({best_row['net_annual_lift'] - alt['net_annual_lift']:+,.0f})")
lines.append(f"  vs random targeting     : "
              f"${rand['net_annual_lift']:,.0f} ({best_row['net_annual_lift'] - rand['net_annual_lift']:+,.0f})")
lines.append("=" * 74)
print("\n".join(lines))

## 10. Proposal statement

> **Recommendation.** Run a targeted retention campaign on the **top 10% of the
> customer base ranked by *revenue-at-risk*** — Model A's churn probability × Model B's
> predicted monthly revenue — rather than on churn probability alone or on customer
> value alone.
>
> **Why two models.** Churners and stayers spend the same, \$58.21 vs \$59.22
> (figure 02), so *value alone points at the wrong people* and *risk alone spends
> money saving cheap customers*. The combined ranking is the only one that targets
> valuable customers who are actually about to leave: the top 10% of it captures
> **32.6% of all revenue lost to churn**, against 17.7% for churn probability alone,
> 26.5% for customer value alone and 10% for random selection (figure 10).
>
> **Quantified impact** at the stated assumptions (figures 11–12), contacting
> 9,964 customers:
>
> | | monthly | annualised |
> |---|---|---|
> | Campaign cost (outreach \$15,942 + discounts \$22,546) | \$38,489 | \$461,862 |
> | Revenue kept from prevented churn | \$80,038 | \$960,451 |
> | **Net revenue lift** | **\$41,549** | **\$498,586** |
> | ROI (net ÷ cost) | **1.08×** | 1.08× |
>
> Versus the same budget spent on churn probability alone the combined score earns
> **+\$302,315 a year (+154%)**, and **+\$469,147 against random targeting**. The
> campaign pays for itself once the offer removes just **~15% of a customer's churn
> risk at a 5% discount** (10% if no discount is given) — the efficacy sweep in
> figure 12 shows the whole profitable region.
>
> **The levers.** Three variables decide where the lift sits (figures 07 and 13):
> **overage exposure** — customers above 15% of minutes outside plan are 14.7% of
> the base but churn at 56.1% versus 48.0% and spend \$98.08 a month versus \$46.25,
> making them both the most valuable and the most fragile group (57% of the
> contacts we would make); **equipment age** — churn climbs from 45.2% under a year
> to 57.9% past two years, and handset age is Model A's strongest feature by gain;
> and **the 1–2 year renewal window** — the largest band (42,868 customers) and the
> highest-churn (51.8%), which tells us *when* to run the campaign. The offer is
> therefore built as **a plan migration that removes the overage bill shock plus a
> handset renewal path for accounts over 18 months of equipment age**, timed to the
> renewal window (figure 13). Usage trend was tested as a fourth lever and
> rejected — the data does not support it.

### Assumptions register (challenge these, not the model)

| # | Assumption | Value | If wrong |
|---|---|---|---|
| 1 | Contact cost | \$2.00 per contact reached | linear effect on cost |
| 2 | Contact rate | 80% of targeted customers reached | scales kept revenue *and* cost |
| 3 | Offer acceptance | 40% of those reached | scales kept revenue *and* discount cost |
| 4 | Discount depth | 5% of the monthly bill while retained | figure 12 shows the full curve |
| 5 | Efficacy | 30% of an accepting customer's churn risk removed | figure 12 shows the full curve |
| 6 | Horizon | 1 month, annualised ×12 | assumes no offer fatigue across cycles |
| 7 | Churn window | dataset churn measured over the observation window (~50% of the base) | at a realistic 10% annual churn the absolute lift shrinks in proportion, the *ranking* effect does not |
| 8 | Prices held constant | no market price change | static-base caveat |

### What goes on the slides

| Slide | Content | Asset |
|---|---|---|
| Problem | churn balance + equal-spend finding | `01`, `02` |
| Data | missingness, correlations | `03`, `04` |
| Signals | engineered features vs churn | `05` |
| Model A | **name / metric / score**, ROC + CM, importance | `06`, `07` |
| Model B | **name / metric / score**, fit, importance | `08`, `09` |
| Prioritisation | gains chart | `10` |
| Impact | lift by scenario, sensitivity | `11`, `12` |
| Levers | segment contribution | `13` |
| Close | recommendation + assumptions register | — |

> **Submission note.** This notebook is already named for submission
> (`ArielDeBrito.ipynb`). Re-run it end-to-end in Google Colab before uploading so
> the outputs are produced in your own environment, then paste the exact text of the
> **MODEL RESULTS** box above onto the model slides — the rubric requires the model
> name, evaluation metric and score written out in words, not shown only as figures.

In [ ]:
# --- inventory of presentation assets -------------------------------------------
from IPython.display import Image, display

files = sorted(ASSETS.glob("*.png"))
print(f"{len(files)} figures in {ASSETS}\n")
for p in files:
    print(f"  {p.name:34s} {p.stat().st_size / 1024:7.0f} KB")

print("\n--- thumbnails ---")
for p in files[:3]:
    display(Image(filename=str(p), width=330))

## 11. References

External sources used in this analysis and in the slide deck (must match the
reference list submitted in Omnicampus):

1. **GSMA (2026).** *The Mobile Economy 2026.* GSMA Association.
   https://www.gsma.com/solutions-and-impact/connectivity-for-good/mobile-economy/
   — report PDF: https://www.gsma.com/solutions-and-impact/connectivity-for-good/mobile-economy/wp-content/uploads/2026/02/The-Mobile-Economy-2026.pdf
   Used for: market context — mobile technologies generated **\$7.6 trillion /
   6.4% of global GDP in 2025**, and **45% of operators name monetisation a
   strategic priority**, i.e. growth now comes from existing subscribers rather
   than new ones.
2. **Gallo, A. (2014).** "The Value of Keeping the Right Customers."
   *Harvard Business Review*, 29 Oct 2014.
   https://hbr.org/2014/10/the-value-of-keeping-the-right-customers
   Used for: the cost asymmetry behind the proposal — acquiring a customer costs
   **5–25× more than retaining one**, and Reichheld's Bain research shows a **5%
   rise in retention lifts profit 25–95%**.
3. **Chen, T. & Guestrin, C. (2016).** *XGBoost: A Scalable Tree Boosting System.*
   Proceedings of the 22nd ACM SIGKDD. https://arxiv.org/abs/1603.02754
   Used for: Model A and Model B algorithm.
4. **Pedregosa et al. (2011).** *Scikit-learn: Machine Learning in Python.* JMLR 12.
   Metrics documentation: https://scikit-learn.org/stable/modules/model_evaluation.html
   Used for: ROC-AUC, accuracy, F1, MAE, RMSE, R².
5. **Company A (client-provided).** `ENG_Company _A_ Dataset Overview.docx` and
   `telecom/Client.csv`, `telecom/Record.csv` — all figures computed in this
   notebook come from these files.

> Every quantitative claim in this notebook is computed from the client dataset.
> The two external figures above (market size, retention economics) carry their own
> citations and must be footnoted on the slide where they appear.